In [1]:
# %pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j

In [6]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-IehlRrXCqUqu3b74km1wT3BlbkFJPZrNRO6xj5UyyPLo9lwq"

In [10]:
# Local host instance variables
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "12345678"

In [8]:
# Neo4J configuration
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph()

In [9]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

llm_transformer = LLMGraphTransformer(llm=llm)

In [13]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import TokenTextSplitter

loader = PyPDFLoader("DeepAnT_A_Deep_Learning_Approach_for_Unsupervised_Anomaly_Detection_in_Time_Series.pdf", extract_images=True)
docs = loader.load()

text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(docs)

In [14]:
llm=ChatOpenAI(temperature=0, model_name="gpt-4o") # gpt-4-0125-preview occasionally has issues
llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(documents)
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [44]:
# # directly show the graph resulting from the given Cypher query
# from neo4j import GraphDatabase
# from yfiles_jupyter_graphs import GraphWidget

# default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

# def showGraph(cypher: str = default_cypher):
#     # create a neo4j session to run queries
#     driver = GraphDatabase.driver(
#         uri = os.environ["NEO4J_URI"],
#         auth = (os.environ["NEO4J_USERNAME"],
#                 os.environ["NEO4J_PASSWORD"]))
#     session = driver.session()
#     widget = GraphWidget(graph = session.run(cypher).graph())
#     widget.node_label_mapping = 'id'
#     #display(widget)
#     return widget

# showGraph()

In [23]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Neo4jVector

# vector_index = Neo4jVector.from_existing_graph(
#     OpenAIEmbeddings(),
#     url = os.environ["NEO4J_URI"],
#     username = os.environ["NEO4J_USERNAME"],
#     password = os.environ["NEO4J_PASSWORD"],
#     search_type="hybrid",
#     node_label="new",
#     text_node_properties=["text"],
#     embedding_node_property="embedding"
# )

vector_index = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(),
    node_label="Document",
    embedding_node_property="embedding",
    text_node_properties=["text"],
    keyword_index_name="document_keyword_index",  # Ensuring consistent index names
    index_name="document_vector_index",           # Ensuring consistent index names
    search_type="hybrid"                          # Specifying search type as hybrid
)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.create.setVectorProperty' has been replaced by 'db.create.setNodeVectorProperty')} {position: line: 1, column: 70, offset: 69} for query: "UNWIND $data AS row MATCH (n:`Document`) WHERE elementId(n) = row.id CALL db.create.setVectorProperty(n, 'embedding', row.embedding) YIELD node RETURN count(*)"


In [27]:
# Retriever
from typing import Tuple, List, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

graph.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, topics, or research entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting topics and research keyword entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

entity_chain = prompt | llm.with_structured_output(Entities)

In [43]:
entity_chain.invoke({"question": "When was this paper is written?"})

Entities(names=['When was this paper is written?'])

In [32]:
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text search.
    It processes the input string by splitting it into words and appending a
    similarity threshold (~2 changed characters) to each word, then combines
    them using the AND operator. Useful for mapping entities from user questions
    to database values, and allows for some misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [33]:
print(structured_retriever("What is TimeSeries?"))

Time_Series - CONTAINS -> Subsequence
Marotta_Mpv-41_Series_Valve - MEASURES -> Time_Series


In [34]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

In [38]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser
# Condense a chat history and follow-up question into a standalone question
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""  # noqa: E501
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [40]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [41]:
chain.invoke({"question": "What is the paper is about?"})

Search query: What is the paper is about?


'The paper discusses anomaly detection techniques, particularly focusing on a method called DeepAnT for detecting anomalies in time series data. It provides a detailed evaluation of DeepAnT, compares it with other state-of-the-art methods, and categorizes various anomaly detection techniques based on supervision levels and underlying methods. The paper also reviews commonly used techniques for point anomalies and those based on deep neural networks.'

In [42]:
chain.invoke(
    {
        "question": "When was this paper written?",
        "chat_history": [("The paper discusses anomaly detection techniques, particularly focusing on a method called DeepAnT for detecting anomalies in time series data. It provides a detailed evaluation of DeepAnT, compares it with other state-of-the-art methods, and categorizes various anomaly detection techniques based on supervision levels and underlying methods. The paper also reviews commonly used techniques for point anomalies and those based on deep neural networks.", "Anomaly Time Series Analysis")],
    }
)

Search query: When was the paper discussing anomaly detection techniques, particularly focusing on a method called DeepAnT, written?


'The paper was written in late 2018, with the date of publication being December 19, 2018.'